# Customizing Your `unichart` Environment with Presets

This notebook shows how to bundle your preferred look-and-feel into a **single
function** that returns a ready-to-use `UnichartNotebook`. Instead of repeating
formatting setup in every analysis, you call `styled_notebook("ocean", data=df)`
and get a notebook that already knows your palette, markers, fonts, and theme.

**What "environment customization" covers** — every knob the preset function sets:

| Setting | Attribute / method | Effect |
|---------|--------------------|--------|
| Dataset palette | `nb.color_map` | colors assigned to datasets by index *at load time* |
| Dataset markers | `nb.marker_map` | marker symbols assigned to datasets by index |
| Theme | `nb.darkmode` | light vs dark Plotly template |
| Title | `nb.suptitle` | default figure title |
| Font sizes | `nb.suptitle_size`, `nb.axes_title_size`, `nb.axes_tick_size`, `nb.legend_size`, `nb.subplot_title_size`, `nb.hover_size` | text sizing across every plot |
| Variable identity color | `nb.var_format(var, color=...)` | per-variable color (used by `by='ymult'` / `by='dataset_x'`) |
| Reference lines / limits | `nb.line(...)`, `nb.axis_limits` | annotations carried onto plots |

> Key detail: `color_map` / `marker_map` are consumed **when data is loaded**, so the
> preset must be applied *before* `load_df`. The function below does exactly that.

## Setup — imports and sample data

In [1]:
# --- make repo-root importable (notebook lives in demo_notebooks/) ---
import sys, os
_repo_root = os.path.abspath(os.path.join(os.getcwd(), os.pardir))
if _repo_root not in sys.path:
    sys.path.insert(0, _repo_root)

import numpy as np
import pandas as pd
import unichart

def sample_data(seed=3):
    "Three 'engine runs', each with an RPM sweep and a few sensor channels."
    rng = np.random.default_rng(seed)
    rows = []
    for eng in ["Engine A", "Engine B", "Engine C"]:
        bias = 1.0 + 0.06 * ["Engine A", "Engine B", "Engine C"].index(eng)
        for phase in ["Idle", "Takeoff", "Cruise", "Descent"]:
            base_rpm = {"Idle": 1800, "Takeoff": 9500, "Cruise": 7200, "Descent": 4000}[phase]
            for _ in range(15):
                rpm = base_rpm * bias + rng.normal(0, 120)
                rows.append({
                    "ENGINE": eng, "PHASE": phase, "RPM": rpm,
                    "Temp": 300 + rpm / 14 + rng.normal(0, 12),
                    "Pressure": 28 + rpm / 350 + rng.normal(0, 1.2),
                    "Vibration": 0.4 + rpm / 12000 + rng.normal(0, 0.04),
                })
    return pd.DataFrame(rows)

df = sample_data()
df.head()

,ENGINE,PHASE,RPM,Temp,Pressure,Vibration
0,Engine A,Idle,2044.910295,415.397041,34.344319,0.547698
1,Engine A,Idle,1745.682085,422.104412,30.563680,0.536196
2,Engine A,Idle,1696.174431,461.031311,33.117157,0.527243
3,Engine A,Idle,1766.245510,418.143837,31.780235,0.531555
4,Engine A,Idle,1857.833447,429.839746,34.457406,0.546827


## The preset function

`THEMES` is the user-editable registry; `styled_notebook` applies a chosen theme's
presets to a fresh `UnichartNotebook` and (optionally) loads data with data-aware
presets layered on top. Everything it does uses the public API.

In [2]:
THEMES = {
    "ocean": {
        "darkmode": False,
        "color_map": ["#0B6E99", "#2BB3C0", "#7FD1AE"],
        "marker_map": ["o", "s", "^"],
        "fonts": {"suptitle_size": 26, "axes_title_size": 16, "axes_tick_size": 12,
                  "legend_size": 13, "subplot_title_size": 15, "hover_size": 12},
    },
    "dark_dashboard": {
        "darkmode": True,
        "color_map": ["#00E5FF", "#FF4081", "#FFD740"],
        "marker_map": ["D", "o", "s"],
        "fonts": {"suptitle_size": 30, "axes_title_size": 18, "axes_tick_size": 13,
                  "legend_size": 14, "subplot_title_size": 16, "hover_size": 13},
    },
    "minimal": {
        "darkmode": False,
        "color_map": ["#222222", "#888888", "#BBBBBB"],
        "marker_map": [".", "o", "s"],
        "fonts": {"suptitle_size": 22, "axes_title_size": 14, "axes_tick_size": 11,
                  "legend_size": 11, "subplot_title_size": 13},
    },
}

# Per-variable identity colors, applied when those columns are present.
VARIABLE_COLORS = {"Temp": "#E45756", "Pressure": "#4C78A8", "Vibration": "#54A24B"}


def styled_notebook(theme="ocean", data=None,
                    set_name_column=None, set_idx_column=None, suptitle=None):
    """Return a UnichartNotebook pre-configured with a formatting preset.

    theme : key into THEMES.
    data  : optional DataFrame; if given it is loaded and data-aware presets
            (variable identity colors, a Temp red-line, axis limit) are applied.
    """
    if theme not in THEMES:
        raise ValueError(f"Unknown theme {theme!r}; choose from {list(THEMES)}")
    preset = THEMES[theme]

    nb = unichart.UnichartNotebook()

    # --- environment-level presets (must precede load_df for the maps to apply) ---
    nb.darkmode = preset["darkmode"]
    nb.color_map = preset["color_map"]
    nb.marker_map = preset["marker_map"]
    nb.suptitle = suptitle if suptitle is not None else f"{theme.replace('_', ' ').title()} preset"
    for attr, size in preset["fonts"].items():
        setattr(nb, attr, size)

    # --- optional data + data-aware presets ---
    if data is not None:
        nb.load_df(data, set_name_column=set_name_column, set_idx_column=set_idx_column)
        present = {c for ds in nb.sets for c in ds.columns}
        for var, col in VARIABLE_COLORS.items():
            if var in present:
                nb.var_format(var, color=col)        # identity color for ymult / dataset_x
        if "Temp" in present:
            nb.axis_limits["Temp"] = [200, 1000]
            nb.line("Temp", 900)                     # shared red-line reference

    return nb

## 1. Ocean preset — palette, markers, and fonts in one call

A single call returns a notebook that's ready to plot. Note the datasets already
carry the ocean palette and marker symbols (assigned at load time from the preset).

In [3]:
nb = styled_notebook("ocean", data=df, set_name_column="ENGINE", set_idx_column="ENGINE")

# the presets are live on the returned object:
print("darkmode   :", nb.darkmode)
print("color_map  :", list(nb.color_map)[:3])
print("dataset cols:", "colors ->", [ds.color for ds in nb.sets],
      "| markers ->", [ds.marker for ds in nb.sets])
print("suptitle_size:", nb.suptitle_size, " axes_title_size:", nb.axes_title_size)

nb.plot(x="RPM", y=["Temp", "Pressure"], by="vars")

UniChart Notebook Environment Initialized.
Loaded Set 0: Engine A
Loaded Set 1: Engine B
Loaded Set 2: Engine C
darkmode   : False
color_map  : ['#0B6E99', '#2BB3C0', '#7FD1AE']
dataset cols: colors -> ['#0B6E99', '#2BB3C0', '#7FD1AE'] | markers -> ['o', 's', '^']
suptitle_size: 26  axes_title_size: 16


## 2. Dark dashboard preset — same data, different environment

Swapping the theme name is the only change. Dark template, vivid palette, larger fonts.

In [4]:
nb_dark = styled_notebook("dark_dashboard", data=df,
                          set_name_column="ENGINE", set_idx_column="ENGINE")
nb_dark.plot(x="RPM", y=["Temp", "Pressure"], by="vars")

UniChart Notebook Environment Initialized.
Loaded Set 0: Engine A
Loaded Set 1: Engine B
Loaded Set 2: Engine C


## 3. Variable identity colors carry into multi-axis & dataset_x views

The preset also set per-variable colors via `var_format`. These show up where
variable color is meaningful — the multi-axis (`by='ymult'`) and `by='dataset_x'`
views — independent of the dataset palette.

In [5]:
nb.plot(x="RPM", y=["Temp", "Pressure", "Vibration"], by="ymult")

In [6]:
nb.suptitle = "Per-variable colors on dataset_x bars"
nb.bar(y=["Temp", "Pressure", "Vibration"], by="dataset_x")

## 4. The preset follows you across plot types

The same environment styles bars, boxes, and histograms too — nothing per-plot to set.

In [7]:
nb_min = styled_notebook("minimal", data=df,
                         set_name_column="ENGINE", set_idx_column="ENGINE")
nb_min.suptitle = "Minimal preset — box plot"
nb_min.box(x="PHASE", y="Temp", by="vars")

UniChart Notebook Environment Initialized.
Loaded Set 0: Engine A
Loaded Set 1: Engine B
Loaded Set 2: Engine C


## Add your own theme

To customize your environment, add an entry to `THEMES` and call `styled_notebook`
with its name:

```python
THEMES["sunrise"] = {
    "darkmode": False,
    "color_map": ["#FF6B35", "#F7C59F", "#EFEFD0"],
    "marker_map": ["o", "D", "^"],
    "fonts": {"suptitle_size": 24, "axes_title_size": 15, "axes_tick_size": 12,
              "legend_size": 12, "subplot_title_size": 14},
}
nb = styled_notebook("sunrise", data=df,
                     set_name_column="ENGINE", set_idx_column="ENGINE")
```

Because the function uses only the public API (`color_map`, `marker_map`, `darkmode`,
the `*_size` font attributes, `var_format`, `axis_limits`, `line`), any preset you
build here behaves identically to formatting set by hand — just packaged for reuse.